In [3]:
!pip install -q --upgrade \
accelerate==0.30.1 \
bitsandbytes \
sentencepiece \
safetensors \
fastapi \
uvicorn \
pyngrok

!pip install -U transformers tokenizers accelerate

!pip install -q peft datasets

!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 111.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 25.8 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 0.30.1
    Uninstalling accelerate-0.30.1:
      Successfully uninstalled accelerate-0.30.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 16.5 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
import torch

# Prevent CUDA fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Enable TF32
torch.backends.cuda.matmul.allow_tf32 = True


# ==============================
# DRIVE PATHS
# ==============================

DRIVE_ROOT = "/content/drive/MyDrive/Goober"

MODEL_DIR = os.path.join(DRIVE_ROOT, "Models")

BASE_MODEL_NAME = "smollm3-3b"

BASE_MODEL_PATH = os.path.join(MODEL_DIR, BASE_MODEL_NAME)

DATASET_PATH = os.path.join(DRIVE_ROOT, "data/final/rrgtrain.jsonl")


# ==============================
# CHECKPOINTS
# ==============================

CHECKPOINT_ROOT = os.path.join(MODEL_DIR, "checkpoints/RRG1")

FINAL_CHECKPOINT = os.path.join(CHECKPOINT_ROOT, "RRGFINAL")

RESUME_CHECKPOINT = None

CHECKPOINT_EVERY_N_STEPS = 350


# ==============================
# TRAINING HYPERPARAMETERS
# ==============================

MAX_SEQ_LENGTH = 160

TRAIN_BATCH_SIZE = 28
GRAD_ACCUM = 2

LEARNING_RATE = 9e-5
NUM_EPOCHS = 2

LOGGING_STEPS = 50


# ==============================
# DEVICE
# ==============================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)
print("Base model:", BASE_MODEL_PATH)
print("Dataset:", DATASET_PATH)

Using device: cuda
Base model: /content/drive/MyDrive/Goober/Models/smollm3-3b
Dataset: /content/drive/MyDrive/Goober/data/final/rrgtrain.jsonl


In [ ]:
import os
import torch
import shutil

from datasets import load_dataset, Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    default_data_collator
)

from peft import LoraConfig, get_peft_model, PeftModel


# =====================================================
# CUDA SETTINGS
# =====================================================

if torch.cuda.is_available():

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(True)


# =====================================================
# COPY DATASET FROM DRIVE → LOCAL SSD
# =====================================================

print("Copying dataset to local runtime...")

LOCAL_DATASET_PATH = "/content/train_dataset.jsonl"

if not os.path.exists(LOCAL_DATASET_PATH):
    shutil.copy(DATASET_PATH, LOCAL_DATASET_PATH)

print("Local dataset ready:", LOCAL_DATASET_PATH)


# =====================================================
# LOAD DATASET
# =====================================================

print("Loading dataset...")

dataset = load_dataset(
    "json",
    data_files=LOCAL_DATASET_PATH,
    split="train"
)

print("Samples:", len(dataset))


# =====================================================
# TOKENIZER
# =====================================================

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_PATH,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


# =====================================================
# BUILD TEXT
# =====================================================

def merge_text(example):
    example["text"] = example["prompt"] + example["completion"]
    return example

dataset = dataset.map(merge_text)


# =====================================================
# TOKENIZE
# =====================================================

print("Tokenizing dataset...")

def tokenize(example):
    return tokenizer(example["text"], truncation=False)

dataset = dataset.map(
    tokenize,
    remove_columns=dataset.column_names
)


# =====================================================
# GLOBAL TOKEN PACKING (NEW)
# =====================================================

print("Packing tokens globally...")

all_tokens = []

for ids in dataset["input_ids"]:
    all_tokens.extend(ids)

total_tokens = len(all_tokens)

usable_length = (total_tokens // MAX_SEQ_LENGTH) * MAX_SEQ_LENGTH
all_tokens = all_tokens[:usable_length]

print("Total tokens:", total_tokens)
print("Usable tokens:", usable_length)

chunks = [
    all_tokens[i:i + MAX_SEQ_LENGTH]
    for i in range(0, usable_length, MAX_SEQ_LENGTH)
]

print("Packed sequences:", len(chunks))


dataset = Dataset.from_dict({
    "input_ids": chunks,
    "labels": [c[:] for c in chunks],
    "attention_mask": [[1] * MAX_SEQ_LENGTH for _ in chunks]
})

dataset.set_format(
    type="torch",
    columns=["input_ids", "labels", "attention_mask"]
)


# =====================================================
# LOAD MODEL (QLORA)
# =====================================================

print("Loading base model...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True
)

model.config.use_cache = False

torch.cuda.empty_cache()


# =====================================================
# GRADIENT CHECKPOINTING
# =====================================================

model.gradient_checkpointing_enable()
model.enable_input_require_grads()


# =====================================================
# LORA
# =====================================================

print("Applying LoRA...")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()


# =====================================================
# RESUME CHECKPOINT
# =====================================================

if RESUME_CHECKPOINT is not None:

    print("Loading adapter checkpoint...")

    model = PeftModel.from_pretrained(
        model,
        RESUME_CHECKPOINT
    )


# =====================================================
# TRAINING ARGS
# =====================================================

training_args = TrainingArguments(

    output_dir=CHECKPOINT_ROOT,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,

    logging_steps=LOGGING_STEPS,

    save_steps=CHECKPOINT_EVERY_N_STEPS,
    save_total_limit=5,

    lr_scheduler_type="cosine",
    warmup_steps=100,

    fp16=True,

    gradient_checkpointing=True,

    dataloader_num_workers=2,

    report_to="none"
)


# =====================================================
# TRAINER
# =====================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=default_data_collator
)


# =====================================================
# TRAIN
# =====================================================

print("Starting training...")

trainer.train(resume_from_checkpoint=RESUME_CHECKPOINT)


# =====================================================
# SAVE
# =====================================================

print("Saving final adapter...")

trainer.model.save_pretrained(FINAL_CHECKPOINT)
tokenizer.save_pretrained(FINAL_CHECKPOINT)

print("Training complete.")

Copying dataset to local runtime...
Local dataset ready: /content/train_dataset.jsonl
Loading dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Samples: 207494
Loading tokenizer...


Map:   0%|          | 0/207494 [00:00<?, ? examples/s]

Tokenizing dataset...


Map:   0%|          | 0/207494 [00:00<?, ? examples/s]

Packing tokens globally...
Total tokens: 13959167
Usable tokens: 13959040
Packed sequences: 87244
Loading base model...


Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

Applying LoRA...
trainable params: 7,667,712 || all params: 3,082,766,336 || trainable%: 0.2487
Starting training...


Step,Training Loss
50,3.039519
100,1.990210
150,1.742887
200,1.682187
